# 🧠 ELF: Diffusion Language Model Architecture, Training & Generation Guide

This notebook is an in-depth guide to understanding the **ELF (Diffusion Language Model)** architecture, its training processes, and generation trajectory.

To keep things simple, we have **removed all distributed computing overhead** (no `pmap`, `pmean`, or device replication), so you can understand the exact mathematical and architectural flow on a single CPU/GPU.

## 1. Key Architectural Features & Training Details

Here are the core concepts of the ELF model as implemented in the codebase:

### 1.1 Self-Conditioning & Double Input Dimension
- **Why double dimension?** As seen in the initialization code:
  `input_dim = 2 * encoder_config.d_model if config.self_cond_prob > 0 else encoder_config.d_model`
- If self-conditioning is enabled (`self_cond_prob > 0`), the model takes a concatenated input of shape `(Batch, Seq_Len, 2 * text_encoder_dim)`. The first half contains the current noisy latent $z_t$, and the second half contains the model's prediction of the clean latent $x_0$ from a previous step.
- Inside the model (`ELF.__call__`), if the input dimension is doubled, a projection layer (`self_cond_proj`) projects it back to the single `text_encoder_dim`:
  ```python
  if x.shape[-1] == 2 * self.text_encoder_dim:
      x = nn.Dense(self.text_encoder_dim, name='self_cond_proj')(x)
  ```

### 1.2 Logit-Normal Timestep Sampling
- Instead of sampling timesteps $t \in [0, 1]$ uniformly during training, ELF defaults to a **Logit-Normal** distribution:
  $$t = \sigma(z), \quad z \sim \mathcal{N}(P_{\text{mean}}, P_{\text{std}})$$
- This biases the training timesteps toward the middle of the trajectory, where the denoising task is most informative, while spending less capacity on extremely noisy ($t \to 0$) or extremely clean ($t \to 1$) states.

### 1.3 Classifier-Free Guidance (CFG) & Attention Mask Label Dropping
- To support Classifier-Free Guidance, with probability `label_drop_prob`, we perform **label dropping** during training to teach the model to make unconditional predictions.
- Crucially, to prevent target tokens from attending to the prompt/condition tokens in dropped samples, we modify the attention mask:
  ```python
  block_mask = (1 - cond_mask)[:, :, None] * cond_mask[:, None, :]
  encoder_attention_mask = encoder_attention_mask * (1 - drop * block_mask)
  ```
  This block mask is 1 only at positions where a target token attempts to attend to a condition token, successfully isolating them.

### 1.4 Factored Unembedding Layer (Decoder Head)
- During training, with probability `decoder_prob`, we execute a decoder step. We noise the continuous latent at $t=1$, pass it to the model, and project the final hidden states to token logits using a factored unembedding head:
  $$\text{Logits} = \text{GELU}(X \cdot W_{\text{proj}} + B_{\text{proj}}) W_{\text{unembed}} + B_{\text{unembed}}$$
- This project-down step reduces parameter cost and aligns the representations with the text encoder's space.

### 1.5 Exponential Moving Average (EMA) Updates
- To stabilize generation, we keep track of an EMA version of the model parameters (`ema_params1`).
- The EMA weights are updated only on actual optimizer steps (not on gradient accumulation steps) using a decay rate $\mu$ (e.g., $0.9999$):
  $$	heta_{\text{EMA}} \leftarrow \mu \cdot \theta_{\text{EMA}} + (1 - \mu) \cdot \theta_{\text{online}}$$

## 2. JAX & Helper Imports

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Dict, Tuple, Any

print("JAX Version:", jax.__version__)

## 3. Logit-Normal Timestep Sampling Simulation

Let's write and visualize the logit-normal distribution used for timestep sampling.

In [ ]:
def sample_timesteps_logit_normal(rng, batch_size, P_mean=-0.8, P_std=0.8):
    """
    Biases timesteps toward the middle via sigmoid(N(P_mean, P_std))
    """
    z = jax.random.normal(rng, (batch_size,)) * P_std + P_mean
    return jax.nn.sigmoid(z)

key = jax.random.PRNGKey(42)
t_samples = sample_timesteps_logit_normal(key, 1000)
print("Mean sampled t:", jnp.mean(t_samples))
print("Min/Max sampled t:", jnp.min(t_samples), jnp.max(t_samples))

## 4. Single-Device Training Step with Self-Conditioning, Label Dropping, and Factored Unembedding

Here is the complete JAX code for a single-device training step containing all model features (excluding multi-device communications).

In [ ]:
def add_noise_single(x0, noise, t, noise_scale):
    t_expanded = t.reshape(-1, 1, 1)
    return t_expanded * x0 + (1.0 - t_expanded) * noise * noise_scale

def compute_velocity_target(x0, z, t, t_eps=0.05):
    t_expanded = t.reshape(-1, 1, 1)
    return (x0 - z) / jnp.maximum(1.0 - t_expanded, t_eps)

def train_step_single_device(
    state: Any,            # TrainState
    encoder_params: Dict,  # Text encoder parameters
    encoder_apply_fn: Any, # Text encoder apply function
    batch: Dict[str, jnp.ndarray],
    config: Any,
    rng: jax.random.PRNGKey,
) -> Tuple[Any, Dict[str, float]]:
    
    # Split RNGs
    t_rng, noise_rng, self_cond_mask_rng, self_cond_val_rng, branch_rng = jax.random.split(rng, 5)
    
    # 1. Attention mask modification for label dropping
    encoder_attention_mask = batch["encoder_attention_mask"]
    if config.label_drop_prob > 0:
        drop = batch["label_drop_mask"][:, None, None]
        cond_mask = batch["cond_seq_mask"]
        # block_mask blocks target tokens from attending to condition tokens
        block_mask = (1.0 - cond_mask)[:, :, None] * cond_mask[:, None, :]
        encoder_attention_mask = encoder_attention_mask * (1.0 - drop * block_mask)
        
    # 2. Encode text tokens to latents x0
    latents = encoder_apply_fn(
        {"params": encoder_params},
        input_ids=batch["input_ids"],
        attention_mask=encoder_attention_mask,
        deterministic=True
    )
    x0 = (latents - config.latent_mean) / config.latent_std
    batch_size, seq_len = x0.shape[0], x0.shape[1]
    
    is_decoder_step = jax.random.bernoulli(branch_rng, config.decoder_prob)
    
    def loss_fn(params):
        # DECODER BRANCH (Cross Entropy loss on vocabulary)
        def _decoder_branch():
            t_ones = jnp.ones((batch_size,))
            noise = jax.random.normal(noise_rng, x0.shape, dtype=x0.dtype)
            
            # Logit-normal noised latent at t=1
            decoder_lambda_t = jax.nn.sigmoid(jax.random.normal(self_cond_val_rng, (batch_size, seq_len, 1)) * 0.8 - 0.8)
            decoder_z = decoder_lambda_t * x0 + (1.0 - decoder_lambda_t) * noise * config.decoder_noise_scale
            
            # If self-conditioning dimension is active, we append a zero-filled state
            if config.self_cond_prob > 0:
                decoder_z = jnp.concatenate([decoder_z, jnp.zeros_like(decoder_z)], axis=-1)
                
            _, decoder_logits = state.apply_fn(
                {"params": params},
                decoder_z, t_ones,
                decoder_step_active=True,
                deterministic=True
            )
            
            log_probs = jax.nn.log_softmax(decoder_logits, axis=-1)
            ce = -jnp.take_along_axis(log_probs, batch["input_ids"][..., None], axis=-1).squeeze(-1)
            
            # Calculate loss only on target tokens
            loss_mask = batch["attention_mask"] * (1.0 - batch["cond_seq_mask"])
            ce_loss = jnp.sum(ce * loss_mask) / jnp.maximum(jnp.sum(loss_mask), 1.0)
            return ce_loss, (jnp.zeros(()), ce_loss)

        # DENOISER BRANCH (L2 loss on flow velocity)
        def _denoiser_branch():
            # Sample timesteps using logit normal
            t = sample_timesteps_logit_normal(t_rng, batch_size, config.denoiser_p_mean, config.denoiser_p_std)
            noise = jax.random.normal(noise_rng, x0.shape, dtype=x0.dtype)
            
            # Interpolate to find noisy state z_t
            z_denoiser = add_noise_single(x0, noise, t, config.denoiser_noise_scale)
            
            # Zero out prompt positions if label drop is activated
            if config.label_drop_prob > 0:
                drop_flag = batch["label_drop_mask"][:, None, None]
                cond_seq_mask = batch["cond_seq_mask"][:, :, None]
                z_denoiser = jnp.where(drop_flag & (cond_seq_mask > 0), jnp.zeros_like(z_denoiser), z_denoiser)
            
            # Self-conditioning computation
            if config.self_cond_prob > 0:
                # With probability self_cond_prob, run online model without grads to get estimated x_pred
                use_self_cond = jax.random.uniform(self_cond_mask_rng, (batch_size,)) < config.self_cond_prob
                use_self_cond_expanded = use_self_cond.reshape(-1, 1, 1).astype(x0.dtype)
                
                # Initial estimation
                z_init = jnp.concatenate([z_denoiser, jnp.zeros_like(z_denoiser)], axis=-1)
                net_out_init, _ = state.apply_fn(
                    {"params": params},
                    z_init, t,
                    decoder_step_active=False,
                    deterministic=True
                )
                net_out_init = jax.lax.stop_gradient(net_out_init)
                x_pred_init = net_out_init
                
                # Mask with self-cond flag
                x_pred_cond = x_pred_init * use_self_cond_expanded
                
                # Final inputs concatenated
                z_input = jnp.concatenate([z_denoiser, x_pred_cond], axis=-1)
            else:
                z_input = z_denoiser
                
            # Forward pass
            predicted_v, _ = state.apply_fn(
                {"params": params},
                z_input, t,
                decoder_step_active=False,
                deterministic=True
            )
            
            v_target = compute_velocity_target(x0, z_denoiser, t, config.t_eps)
            
            # L2 Loss on velocity
            per_dim_l2 = (predicted_v - v_target) ** 2
            l2_loss = jnp.mean(per_dim_l2, axis=-1)
            
            loss_mask = batch["attention_mask"] * (1.0 - batch["cond_seq_mask"])
            denoiser_loss = jnp.sum(l2_loss * loss_mask) / jnp.maximum(jnp.sum(loss_mask), 1.0)
            return denoiser_loss, (denoiser_loss, jnp.zeros(()))
            
        return jax.lax.cond(is_decoder_step, _decoder_branch, _denoiser_branch)
        
    (loss, (l2_loss, ce_loss)), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    
    # Update weights
    new_state = state.apply_gradients(grads=grads)
    
    # Update EMA parameters (decay * ema_params + (1 - decay) * new_params)
    def ema_update(ema, current):
        return jax.tree_util.tree_map(lambda e, c: e * config.ema_decay1 + c * (1.0 - config.ema_decay1), ema, current)
    
    new_ema_params = ema_update(state.ema_params1, new_state.params)
    new_state = new_state.replace(ema_params1=new_ema_params)
    
    metrics = {
        "loss": loss,
        "l2_loss": l2_loss,
        "ce_loss": ce_loss,
    }
    return new_state, metrics

## 5. Generative Trajectory & Sampling Pipeline

The sampling process integrates forward from $t=0.0 \to 1.0$ using predicted velocities. Here is the single-device pipeline for both Unconditional and Conditional (Infilling) generation.

In [ ]:
def restore_cond(z, cond_seq, cond_seq_mask):
    """
    Clamps prompt token positions in latent space to their true encoded values.
    """
    mask = cond_seq_mask[..., None]
    return mask * cond_seq + (1.0 - mask) * z

def run_sampling(
    model_apply_fn,
    params,
    z_start: jnp.ndarray,     # Initial noise
    timesteps: jnp.ndarray,   # Timesteps sequence (0.0 to 1.0)
    cond_seq: jnp.ndarray = None,
    cond_seq_mask: jnp.ndarray = None,
    config: Any = None,
) -> jnp.ndarray:
    
    z = z_start
    batch_size = z.shape[0]
    x_pred = jnp.zeros_like(z)
    
    is_conditional = cond_seq is not None and cond_seq_mask is not None
    if is_conditional:
        z = restore_cond(z, cond_seq, cond_seq_mask)
        
    # ODE Euler Integration loop
    for i in range(len(timesteps) - 1):
        t_curr = timesteps[i]
        t_next = timesteps[i+1]
        t_batch = jnp.full((batch_size,), t_curr)
        
        # If self-conditioning is enabled, concatenate previous step prediction
        if config.self_cond_prob > 0:
            z_input = jnp.concatenate([z, x_pred], axis=-1)
        else:
            z_input = z
            
        # Forward pass to predict v
        net_out, _ = model_apply_fn(
            {"params": params},
            z_input, t_batch,
            decoder_step_active=False,
            deterministic=True
        )
        
        v_pred = (net_out - z) / jnp.maximum(1.0 - t_curr, config.t_eps)
        x_pred = net_out
        
        # Euler step update
        dt = t_next - t_curr
        z = z + dt * v_pred
        
        # Clamp conditional sequence
        if is_conditional:
            z = restore_cond(z, cond_seq, cond_seq_mask)
            x_pred = restore_cond(x_pred, cond_seq, cond_seq_mask)
            
    return z

## 6. Decoding Final Latents into Suffix Tokens

At $t=1$, we run the factored decoder head and select tokens via `argmax`.

In [ ]:
def decode_latents_to_tokens(model_apply_fn, params, z_final, config) -> jnp.ndarray:
    batch_size = z_final.shape[0]
    t_final = jnp.ones((batch_size,))
    
    # If self-conditioning is enabled, append zero-filled state
    if config.self_cond_prob > 0:
        z_input = jnp.concatenate([z_final, jnp.zeros_like(z_final)], axis=-1)
    else:
        z_input = z_final
        
    _, decoder_logits = model_apply_fn(
        {"params": params},
        z_input, t_final,
        decoder_step_active=True,
        deterministic=True
    )
    
    return jnp.argmax(decoder_logits, axis=-1)